In [1]:
%load_ext autoreload
%autoreload 2
%autosave 30

Autosaving every 30 seconds


# Predict with the Flow Matching Model or CGM

In [2]:
import importlib

import hydra
import lightning as L
import numpy as np
import pandas as pd
import torch
import xarray as xr
from einops import rearrange, reduce
from omegaconf import DictConfig
from tqdm import tqdm

from genpp import BASE_DIR
from genpp.configs import add_y_kwargs, del_key, register_resolvers
from genpp.data import FORECAST_ENS_FLAT_AGG_PATH, OBSERVATIONS_FLAT_PATH
from genpp.eval.utils import (
    compute_scores_per_leadtime,
    log_scores,
    save_predictions_dataarray,
    save_scores_df,
    update_wandb_run,
)
from genpp.models.cgm.chen import CNNChenModel
from genpp.models.loss import EnergyScore, EnsembleCRPS, VariogramScore

try:
    register_resolvers()
except Exception:
    pass

torch.set_float32_matmul_precision("high")

## Best Models
FM model: `feik/genpp/blkpcik8`

Chen model: `feik/genpp/qbuvhf5p`


In [3]:
# Model ID is generated by WandB
RUN_PATH = "feik/genpp/blkpcik8"
MODEL_ID = RUN_PATH.split("/")[-1]
OUTPUT_DIR = BASE_DIR.parent.parent / "outputs"

MODEL_DIR = list(OUTPUT_DIR.rglob(f"*{MODEL_ID}*"))[0].parent.parent.parent
MODEL_CHECKPOINT = list(MODEL_DIR.rglob("*.ckpt"))[0]

SCORE_FILE = MODEL_DIR / "scores.csv"

In [4]:
with hydra.initialize_config_dir(config_dir=str(MODEL_DIR / ".hydra"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="config",
    )
# Do not shuffle any dataloader
cfg.data.module.dataloader_config.train.shuffle = False
cfg.data.module.dataloader_config.val.shuffle = False
cfg.data.module.dataloader_config.val.batch_size = 16  # For faster predictions
cfg.data.module.dataloader_config.test.shuffle = False
add_y_kwargs(
    cfg, y_kwargs={"batch_dims": {}, "input_dims": {"feature": 2, "longitude": 37, "latitude": 31}}
)
del_key(cfg.data.module.dataset_config.train.x_kwargs.batch_dims, "time")
del_key(cfg.data.module.dataset_config.train.x_kwargs.batch_dims, "prediction_timedelta")

datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup(stage="validate")
datamodule.setup(stage="test")

Adding y_kwargs to dataset_config.train
Deleted key 'time' from config.
Deleted key 'prediction_timedelta' from config.
Configuration hash: 980620c515fdadefc42dff7cfb4e72f8827ffcc7ce573927f07acac73dffa36d
Cached tensor data found. Verifying configuration...
Using cached tensor data from /home/feik/GenPP/src/genpp/data/weatherbench2/cache/tensor_980620c515fdadefc42dff7cfb4e72f8827ffcc7ce573927f07acac73dffa36d.pt.


In [5]:
class_path = cfg.model._target_

module_path, class_name = class_path.rsplit(".", 1)
module = importlib.import_module(module_path)
ModelClass = getattr(module, class_name)

if ModelClass is CNNChenModel:
    model = ModelClass.load_from_checkpoint(
        MODEL_CHECKPOINT,
        final_activation=hydra.utils.instantiate(cfg.model.final_activation),
        loss_fn=hydra.utils.instantiate(cfg.model.loss_fn),
    )
else:
    model = ModelClass.load_from_checkpoint(MODEL_CHECKPOINT)

Loading scale_variance_td from checkpoint


In [6]:
limit_predict_batches = None
trainer = L.Trainer(
    logger=False, accelerator="gpu", devices=[1], limit_predict_batches=limit_predict_batches
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
test_scores = trainer.test(model, datamodule.test_dataloader())

In [7]:
pred_list = trainer.predict(model, datamodule.val_dataloader(), return_predictions=True)
predictions = torch.cat(pred_list, dim=0)  # type: ignore

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 227/227 [12:30<00:00,  0.30it/s]


In [8]:
# Rescale the predictions (TODO this should happen in the predict step)
reverse_transform = datamodule.y_reverseModules[0]
mean = rearrange(reverse_transform.mean, "f -> 1 1 f 1 1")
scale = rearrange(reverse_transform.scale, "f -> 1 1 f 1 1")

predictions_rescaled = predictions * scale + mean

In [9]:
x_ds = xr.open_dataset(FORECAST_ENS_FLAT_AGG_PATH)

time_slice = {
    "train": hydra.utils.instantiate(cfg.data.splits.train),
    "val": hydra.utils.instantiate(cfg.data.splits.val),
    "test": hydra.utils.instantiate(cfg.data.splits.test),
}
val_slice = time_slice["val"]

x_ds = x_ds.stack(sample=("time", "prediction_timedelta"))


datamodule.cache_metadata

# Now lets get the actual times we need for the val set and and get the val data
init_times = datamodule.cache_metadata["feature_metadata"]["time"]["val"]
timedeltas = datamodule.cache_metadata["feature_metadata"]["prediction_timedelta"]["val"]
val_times = init_times + timedeltas
val_prediction = pd.MultiIndex.from_arrays(
    [init_times, timedeltas], names=["time", "prediction_timedelta"]
)

# Now load the y_val data for the val times
y_val = (
    xr.open_dataset(OBSERVATIONS_FLAT_PATH)
    .sel(time=val_times)
    .to_dataarray("feature")
    .transpose("time", "feature", "longitude", "latitude")
    .rename({"time": "prediction_time"})
    .assign_coords(
        prediction=("prediction_time", val_prediction),
    )
    .swap_dims({"prediction_time": "prediction"})
)
feature_order = list(cfg.data.y_select_variables)
y_val = y_val.sel(feature=feature_order)
y_t = torch.from_numpy(y_val.values).to(predictions_rescaled)

In [10]:
SKIP_VARIOGRAM = False  # Option to skip variogram score computation due to long computation times

# Compute the scores
crps_ens = EnsembleCRPS()
es = EnergyScore(clamp=False)
vs = VariogramScore(p=0.5)

crps_per_margin = crps_ens(predictions_rescaled, y_t)

# Rearrange so that we compute the scores seperatly for each variable
x_spatial = rearrange(predictions_rescaled, "t n d lat lon -> t d n (lat lon)")
y_spatial = rearrange(y_t, "t d lat lon -> t d (lat lon)")

# _u is for un-reduced scores
energy_score_per_var_u = es(x_spatial, y_spatial)

# For the VS there are huge intermediary results, thats why it is computed batchwise
if not SKIP_VARIOGRAM:
    vss = []
    for x_i, y_i in tqdm(zip(x_spatial, y_spatial), total=predictions_rescaled.shape[0]):
        vss.append(vs(x_i, y_i))
    variogram_score_per_var_u = torch.stack(vss)


# Rearrange to compute scores for both variables together
x_full = rearrange(predictions_rescaled, "t n d lat lon -> t n (d lat lon)")
y_full = rearrange(y_t, "t d lat lon -> t (d lat lon)")

energy_score_full_u = es(x_full, y_full)
if not SKIP_VARIOGRAM:
    vss = []
    for x_i, y_i in tqdm(zip(x_full, y_full), total=predictions_rescaled.shape[0]):
        vss.append(vs(x_i, y_i))
    variogram_score_full_u = torch.stack(vss)

100%|██████████| 3620/3620 [36:57<00:00,  1.63it/s]


In [11]:
# Reduce the scores
crps_per_var = reduce(crps_per_margin, "t d h w -> d", reduction="mean")
crps_full = reduce(crps_per_margin, "t d h w -> 1", "mean")
energy_score_per_var = reduce(energy_score_per_var_u, "t d -> d", "mean")
energy_score_full = reduce(energy_score_full_u, "t -> 1", "mean")
if not SKIP_VARIOGRAM:
    variogram_score_per_var = reduce(variogram_score_per_var_u, "t d -> d", "mean")
    variogram_score_full = reduce(variogram_score_full_u, "t -> 1", "mean")

In [12]:
# Log the Scores
model_class = cfg.model._target_.split(".")[-1]


log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="CRPS",
    variables=datamodule.y_select_variables,
    scores=crps_per_var,
)
log_scores(
    file=SCORE_FILE, model=model_class, metric="CRPS", variables=["combined"], scores=crps_full
)
log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="EnergyScore",
    variables=datamodule.y_select_variables,
    scores=energy_score_per_var,
)
log_scores(
    file=SCORE_FILE,
    model=model_class,
    metric="EnergyScore",
    variables=["combined"],
    scores=energy_score_full,
)
if not SKIP_VARIOGRAM:
    log_scores(
        file=SCORE_FILE,
        model=model_class,
        metric="VariogramScore",
        variables=datamodule.y_select_variables,
        scores=variogram_score_per_var,
    )
    # Log full scores
    log_scores(
        file=SCORE_FILE,
        model=model_class,
        metric="VariogramScore",
        variables=["combined"],
        scores=variogram_score_full,
    )

In [13]:
scores = compute_scores_per_leadtime(
    timedeltas,
    crps_per_margin,
    energy_score_per_var_u,
    energy_score_full_u,
    variogram_score_per_var_u if not SKIP_VARIOGRAM else None,
    variogram_score_full_u if not SKIP_VARIOGRAM else None,
    method=None,
)
scores

Processing leadtime 24h with 728 samples
Processing leadtime 48h with 726 samples
Processing leadtime 72h with 724 samples
Processing leadtime 96h with 722 samples
Processing leadtime 120h with 720 samples


{'CRPS_combined': {'24h': 0.40934693813323975,
  '48h': 0.49114203453063965,
  '72h': 0.5865655541419983,
  '96h': 0.71030193567276,
  '120h': 0.8413325548171997},
 'CRPS_2m_temperature': {'24h': 0.44856148958206177,
  '48h': 0.5392646193504333,
  '72h': 0.644510805606842,
  '96h': 0.7803880572319031,
  '120h': 0.9435579776763916},
 'CRPS_10m_windspeed': {'24h': 0.3701324760913849,
  '48h': 0.44301918148994446,
  '72h': 0.5286206603050232,
  '96h': 0.6402161121368408,
  '120h': 0.739107608795166},
 'EnergyScore_combined': {'24h': 25.882959365844727,
  '48h': 30.98809242248535,
  '72h': 36.79679870605469,
  '96h': 44.28303527832031,
  '120h': 52.17192840576172},
 'EnergyScore_2m_temperature': {'24h': 19.714508056640625,
  '48h': 23.46076011657715,
  '72h': 27.817644119262695,
  '96h': 33.258445739746094,
  '120h': 39.59914779663086},
 'EnergyScore_10m_windspeed': {'24h': 16.34520721435547,
  '48h': 19.56443214416504,
  '72h': 23.15732192993164,
  '96h': 27.83060646057129,
  '120h': 31.9

In [14]:
# Update WandB run
full_scores = {"val": scores}
update_wandb_run(f"feik/genpp/{MODEL_ID}", full_scores)

# Also log as a table
records = []
for dataset, metrics in full_scores.items():
    for metric_name, horizons in metrics.items():
        for horizon, value in horizons.items():
            records.append((f"{model.__class__.__name__}", dataset, metric_name, horizon, value))
df = pd.DataFrame(records, columns=["method", "dataset", "metric", "horizon", "value"])

save_scores_df(df=df, run_path=f"feik/genpp/{MODEL_ID}")

wandb: Currently logged in as: feik to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,80
lr-AdamW,0.0001
train_loss,0.74218
trainer/global_step,110483
val_loss,0.79352
val_loss_var_0,0.66716
val_loss_var_1,0.91988


In [15]:
# Save the predictions as a DataArray
if hasattr(model, "n_samples_train"):
    N = np.arange(model.n_samples_train)
elif hasattr(model, "n_samples"):
    N = np.arange(model.n_samples)
else:
    raise ValueError("Model has no attribute n_samples or n_samples_train")

res = xr.DataArray(
    predictions_rescaled.cpu().numpy(),
    coords={
        "prediction": y_val.prediction,
        "sample": N,
        "feature": y_val.feature,
        "longitude": y_val.longitude,
        "latitude": y_val.latitude,
    },
    dims=("prediction", "sample", "feature", "longitude", "latitude"),
)
save_predictions_dataarray(
    predictions=res, save_path=MODEL_DIR / "val_predictions.zarr", overwrite=True
)

Saved predictions to /home/feik/GenPP/outputs/FM_UNET/2025-10-20_19-54-35/val_predictions.zarr.
